# `Dynamic AI Knowledge Assistant`

In [1]:
# ------- Install Dependencies ----------

import sys
!{sys.executable} -m pip install pandas requests tqdm

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ------- Imports & Configurations  ----------

from pathlib import Path
import time
import re
import hashlib
import requests
import pandas as pd
from tqdm.auto import tqdm

# =====================================================
# Project Configuration
# =====================================================

PROJECT_ROOT = Path.cwd()
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw" / "wikipedia"
METADATA_DIR = PROJECT_ROOT / "data" / "metadata"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================
# Wikipedia API Configuration
# =====================================================

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
HEADERS = {
    "User-Agent": (
        "DynamicAIKnowledgeAssistant/1.0 "
        "(Educational RAG Project)"
    )
}

# =====================================================
# Search / Download Configuration
# =====================================================

SEARCH_LIMIT = 10
MAX_RETRIES = 5
INITIAL_BACKOFF = 2
REQUEST_DELAY = 1.0

C:\Users\RAaM\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =====================================================
# User-defined Knowledge Domains
# =====================================================

KEYWORDS = [
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Large Language Model",
    "Natural Language Processing",
    "Data Science",
    "Computer Vision",
    "Generative AI",
    "Transformer",
    "Neural Network",
    "Prompt Engineering",
    "Retrieval Augmented Generation",
    "Semantic Search",
    "Vector Database",
    "Feature Engineering"
]

print(f"Number of keywords: {len(KEYWORDS)}")

Number of keywords: 15


In [4]:
# =====================================================
# Validate Keywords
# =====================================================

def validate_keywords(keywords):

    if not isinstance(keywords, list):
        raise TypeError("KEYWORDS must be provided as a list.")

    cleaned_keywords = []

    for keyword in keywords:

        if not isinstance(keyword, str):
            continue

        keyword = keyword.strip()

        if keyword:
            cleaned_keywords.append(keyword)

    # Remove duplicate keywords while preserving order
    cleaned_keywords = list(dict.fromkeys(cleaned_keywords))

    if len(cleaned_keywords) < 5:
        raise ValueError(
            "At least 5 unique keywords are required."
        )

    return cleaned_keywords


KEYWORDS = validate_keywords(KEYWORDS)

print("Validated keywords:")
for i, keyword in enumerate(KEYWORDS, start=1):
    print(f"{i:02d}. {keyword}")

Validated keywords:
01. Artificial Intelligence
02. Machine Learning
03. Deep Learning
04. Large Language Model
05. Natural Language Processing
06. Data Science
07. Computer Vision
08. Generative AI
09. Transformer
10. Neural Network
11. Prompt Engineering
12. Retrieval Augmented Generation
13. Semantic Search
14. Vector Database
15. Feature Engineering


In [5]:
# =====================================================
# Improved Wikipedia API Request
# =====================================================

def wikipedia_request(params):

    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            response = requests.get(
                WIKIPEDIA_API_URL,
                params=params,
                headers=HEADERS,
                timeout=30
            )

            print(
                f"Attempt {attempt}/{MAX_RETRIES} | "
                f"HTTP {response.status_code}"
            )

            # -----------------------------------------
            # Rate limiting
            # -----------------------------------------

            if response.status_code == 429:

                retry_after = response.headers.get(
                    "Retry-After"
                )

                if retry_after:
                    wait_time = int(retry_after)
                else:
                    wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))

                print(
                    f"Rate limited. Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)
                continue

            # -----------------------------------------
            # Server errors
            # -----------------------------------------

            if response.status_code in [500, 502, 503, 504]:

                wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))

                print(
                    f"Server error {response.status_code}. "
                    f"Waiting {wait_time} seconds..."
                )

                time.sleep(wait_time)
                continue

            # -----------------------------------------
            # Other HTTP errors
            # -----------------------------------------

            response.raise_for_status()

            # -----------------------------------------
            # Validate response type
            # -----------------------------------------

            content_type = response.headers.get(
                "Content-Type",
                ""
            )

            if "json" not in content_type.lower():

                raise ValueError(
                    f"Expected JSON but received "
                    f"{content_type}"
                )

            return response.json()

        except requests.exceptions.Timeout as e:

            last_error = e

            wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))

            print(
                f"Timeout. Waiting {wait_time} seconds..."
            )

            time.sleep(wait_time)

        except requests.exceptions.ConnectionError as e:

            last_error = e

            wait_time = INITIAL_BACKOFF * (2 ** (attempt - 1))

            print(
                f"Connection error. "
                f"Waiting {wait_time} seconds..."
            )

            time.sleep(wait_time)

        except requests.exceptions.RequestException as e:

            last_error = e

            print(
                f"Request error: {e}"
            )

            break

        except ValueError as e:

            last_error = e

            print(
                f"Response error: {e}"
            )

            break

    raise RuntimeError(
        f"Wikipedia request failed after "
        f"{MAX_RETRIES} attempts. "
        f"Last error: {last_error}"
    )

In [6]:
# =====================================================
# Search Wikipedia
# =====================================================

def search_articles(keyword, limit=10):

    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "srlimit": limit,
        "format": "json"
    }

    data = wikipedia_request(params)

    results = data.get("query", {}).get("search", [])

    titles = [
        result["title"]
        for result in results
        if result.get("title")
    ]

    time.sleep(REQUEST_DELAY)

    return titles

In [7]:
# =====================================================
# Collect Wikipedia Articles
# =====================================================

all_articles = []

for keyword in KEYWORDS:

    print(f"\nSearching for: {keyword}")

    try:
        titles = search_articles(
            keyword,
            limit=SEARCH_LIMIT
        )

        for title in titles:
            all_articles.append({
                "category": keyword,
                "title": title
            })

        print(f"Found: {len(titles)} articles")

    except Exception as e:
        print(f"Search failed for '{keyword}': {e}")

print("\nTotal search results:", len(all_articles))


Searching for: Artificial Intelligence


Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Machine Learning
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Deep Learning
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Large Language Model
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Natural Language Processing
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Data Science
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Computer Vision
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Generative AI
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Transformer
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Neural Network
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Prompt Engineering
Attempt 1/5 | HTTP 429
Rate limited. Waiting 21 seconds...
Attempt 2/5 | HTTP 200
Found: 10 articles

Searching for: Retrieval Augmented Generation
Attempt 1/5 | HTTP 200
Found: 10 articles

Searching for: Semantic Search
Attempt 1/5 | HTTP 200
Found:

In [8]:
# =====================================================
# Remove Duplicate Articles
# =====================================================

articles_df = pd.DataFrame(all_articles)

print("Before duplicate removal:", len(articles_df))

articles_df = articles_df.drop_duplicates(
    subset=["title"]
).reset_index(drop=True)

print("After duplicate removal:", len(articles_df))

print("\nSample:")
display(articles_df.head(10))

Before duplicate removal: 150
After duplicate removal: 127

Sample:


,category,title
0,Artificial Intelligence,Artificial intelligence
1,Artificial Intelligence,Artificial general intelligence
2,Artificial Intelligence,A.I. Artificial Intelligence
3,Artificial Intelligence,Timeline of artificial intelligence
4,Artificial Intelligence,Artificial Intelligence Act
5,Artificial Intelligence,History of artificial intelligence
6,Artificial Intelligence,Glossary of artificial intelligence
7,Artificial Intelligence,Generative AI
8,Artificial Intelligence,Philosophy of artificial intelligence
9,Artificial Intelligence,Hallucination (artificial intelligence)


In [9]:
# =====================================================
# Safe Filename Generator
# =====================================================

def create_safe_filename(title):

    filename = re.sub(
        r'[<>:"/\\|?*]',
        '',
        title
    )

    filename = re.sub(
        r'\s+',
        '_',
        filename.strip()
    )

    return filename[:150]

In [10]:
# =====================================================
# Content Hash
# =====================================================

def generate_content_hash(text):

    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()

In [11]:
# =====================================================
# Download Wikipedia Article
# =====================================================

def download_article(title):

    params = {
        "action": "query",
        "prop": "extracts|info",
        "explaintext": True,
        "inprop": "url",
        "titles": title,
        "format": "json"
    }

    data = wikipedia_request(params)

    pages = data.get(
        "query",
        {}
    ).get(
        "pages",
        {}
    )

    if not pages:
        raise ValueError(
            f"No page data returned for: {title}"
        )

    page = next(iter(pages.values()))

    if "missing" in page:
        raise ValueError(
            f"Wikipedia page not found: {title}"
        )

    text = page.get("extract", "").strip()

    if not text:
        raise ValueError(
            f"Empty article content: {title}"
        )

    return {
        "title": page.get("title", title),
        "url": page.get(
            "fullurl",
            f"https://en.wikipedia.org/wiki/{title.replace(' ', '_')}"
        ),
        "text": text
    }

In [12]:
# =====================================================
# Metadata and Error Tracking
# =====================================================

metadata_records = []
error_records = []

downloaded_hashes = set()

print("Metadata tracking initialized.")

Metadata tracking initialized.


In [13]:
# =====================================================
# Download, Validate, Cache and Save Article
# =====================================================

def download_and_save_article(title, category):

    try:

        # ---------------------------------------------
        # Determine expected filename
        # ---------------------------------------------

        filename = create_safe_filename(title)
        file_path = OUTPUT_DIR / f"{filename}.txt"

        # ---------------------------------------------
        # Check if article already exists locally
        # ---------------------------------------------

        if file_path.exists():

            text = file_path.read_text(
                encoding="utf-8"
            ).strip()

            if text:

                content_hash = generate_content_hash(text)

                # Avoid duplicate metadata records
                if content_hash not in downloaded_hashes:

                    downloaded_hashes.add(content_hash)

                    metadata_records.append({
                        "article_id": content_hash[:16],
                        "title": title,
                        "url": (
                            "https://en.wikipedia.org/wiki/"
                            + title.replace(" ", "_")
                        ),
                        "category": category,
                        "file_name": file_path.name,
                        "word_count": len(text.split()),
                        "character_count": len(text),
                        "content_hash": content_hash,
                        "download_date": pd.Timestamp.now(),
                        "status": "cached"
                    })

                return {
                    "status": "cached",
                    "title": title
                }

        # ---------------------------------------------
        # Download article
        # ---------------------------------------------

        article = download_article(title)

        text = article["text"]

        # ---------------------------------------------
        # Validate content
        # ---------------------------------------------

        if len(text.strip()) < 100:

            raise ValueError(
                "Article contains insufficient text."
            )

        # ---------------------------------------------
        # Generate content hash
        # ---------------------------------------------

        content_hash = generate_content_hash(text)

        # ---------------------------------------------
        # Check duplicate content
        # ---------------------------------------------

        if content_hash in downloaded_hashes:

            return {
                "status": "duplicate",
                "title": title
            }

        # ---------------------------------------------
        # Save article
        # ---------------------------------------------

        filename = create_safe_filename(
            article["title"]
        )

        file_path = OUTPUT_DIR / f"{filename}.txt"

        file_path.write_text(
            text,
            encoding="utf-8"
        )

        downloaded_hashes.add(content_hash)

        # ---------------------------------------------
        # Create metadata
        # ---------------------------------------------

        metadata_records.append({
            "article_id": content_hash[:16],
            "title": article["title"],
            "url": article["url"],
            "category": category,
            "file_name": file_path.name,
            "word_count": len(text.split()),
            "character_count": len(text),
            "content_hash": content_hash,
            "download_date": pd.Timestamp.now(),
            "status": "success"
        })

        return {
            "status": "success",
            "title": article["title"]
        }

    except Exception as e:

        error_records.append({
            "title": title,
            "category": category,
            "error": str(e),
            "timestamp": pd.Timestamp.now()
        })

        return {
            "status": "failed",
            "title": title,
            "error": str(e)
        }

In [14]:
# =====================================================
# Build Wikipedia Knowledge Base
# =====================================================

start_time = time.time()

total = len(articles_df)

print(
    f"Starting download of {total} unique articles...\n"
)

for row in tqdm(
    articles_df.itertuples(index=False),
    total=total,
    desc="Downloading articles"
):

    download_and_save_article(
        title=row.title,
        category=row.category
    )

    time.sleep(REQUEST_DELAY)

elapsed_time = time.time() - start_time

print("\nDownload completed.")
print(
    f"Time taken: {elapsed_time / 60:.2f} minutes"
)

Starting download of 127 unique articles...



Attempt 1/5 | HTTP 200


Attempt 1/5 | HTTP 200


Attempt 1/5 | HTTP 200


Attempt 1/5 | HTTP 200


Attempt 1/5 | HTTP 200


... (119 outputs left out)

Attempt 1/5 | HTTP 200


Attempt 1/5 | HTTP 429
Rate limited. Waiting 36 seconds...
Attempt 2/5 | HTTP 200


Attempt 1/5 | HTTP 200



Download completed.
Time taken: 12.86 minutes


In [15]:
# =====================================================
# Save Metadata
# =====================================================

metadata_df = pd.DataFrame(
    metadata_records
)

metadata_columns = [
    "article_id",
    "title",
    "url",
    "category",
    "file_name",
    "word_count",
    "character_count",
    "content_hash",
    "download_date",
    "status"
]

metadata_df = metadata_df[
    metadata_columns
]

metadata_path = (
    METADATA_DIR / "metadata.csv"
)

metadata_df.to_csv(
    metadata_path,
    index=False
)

print("Metadata saved to:")
print(metadata_path)

print("Metadata shape:", metadata_df.shape)

display(metadata_df.head())

Metadata saved to:
c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\notebooks\data\metadata\metadata.csv
Metadata shape: (127, 10)


,article_id,title,url,category,file_name,word_count,character_count,content_hash,download_date,status
0,6d0b0f8bd20c7395,Artificial intelligence,https://en.wikipedia.org/wiki/Artificial_intel...,Artificial Intelligence,Artificial_intelligence.txt,13191,86958,6d0b0f8bd20c7395440fc63c223c1c25cc64928d669b1b...,2026-08-14 20:03:14.250954,success
1,ab62a0b9df177332,Artificial general intelligence,https://en.wikipedia.org/wiki/Artificial_gener...,Artificial Intelligence,Artificial_general_intelligence.txt,5602,35821,ab62a0b9df177332b4021cb28d350c4860dec24d2ed433...,2026-08-14 20:03:16.717379,success
2,2ab4c7766d058096,A.I. Artificial Intelligence,https://en.wikipedia.org/wiki/A.I._Artificial_...,Artificial Intelligence,A.I._Artificial_Intelligence.txt,4226,26064,2ab4c7766d058096b278f433fcbdb95269db40aedcac3e...,2026-08-14 20:03:19.676300,success
3,a791e34cec56e201,Timeline of artificial intelligence,https://en.wikipedia.org/wiki/Timeline_of_arti...,Artificial Intelligence,Timeline_of_artificial_intelligence.txt,963,7052,a791e34cec56e201dc68d8948dc38e92922370afcabe77...,2026-08-14 20:03:21.921232,success
4,aa83d961e47969b4,Artificial Intelligence Act,https://en.wikipedia.org/wiki/Artificial_Intel...,Artificial Intelligence,Artificial_Intelligence_Act.txt,3360,22894,aa83d961e47969b4534df52e3450b97ce04baa631c89ce...,2026-08-14 20:03:24.366425,success


In [16]:
# =====================================================
# Save Error Log
# =====================================================

error_df = pd.DataFrame(
    error_records
)

error_path = (
    METADATA_DIR / "error_log.csv"
)

error_df.to_csv(
    error_path,
    index=False
)

print("Error log saved to:")
print(error_path)

print("\nNumber of errors:", len(error_df))

Error log saved to:
c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\notebooks\data\metadata\error_log.csv

Number of errors: 0


In [17]:
# =====================================================
# Archive Stale Articles
# =====================================================

ARCHIVE_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "wikipedia_archive"
)

ARCHIVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Archive directory:")
print(ARCHIVE_DIR)

# =====================================================
# Move Stale Files To Archive
# =====================================================

stale_files = [
    "International_Conference_on_Computer_Vision.txt",
    "MariaDB.txt",
    "Solid-state_transformer.txt"
]

for file_name in stale_files:

    source = OUTPUT_DIR / file_name
    destination = ARCHIVE_DIR / file_name

    if source.exists():

        source.rename(destination)

        print(
            f"Archived: {file_name}"
        )

    else:

        print(
            f"Not found: {file_name}"
        )

Archive directory:
c:\Users\RAaM\OneDrive\Desktop\Dynamic AI Knowledge Assistant\notebooks\data\raw\wikipedia_archive
Archived: International_Conference_on_Computer_Vision.txt
Not found: MariaDB.txt
Not found: Solid-state_transformer.txt


In [18]:
# =====================================================
# Final Knowledge Base Validation
# =====================================================

txt_files = list(
    OUTPUT_DIR.glob("*.txt")
)

txt_file_names = {
    file.name
    for file in txt_files
}

metadata_file_names = set(
    metadata_df["file_name"].astype(str)
)

files_without_metadata = (
    txt_file_names - metadata_file_names
)

metadata_without_files = (
    metadata_file_names - txt_file_names
)

total_words = (
    metadata_df["word_count"].sum()
    if not metadata_df.empty
    else 0
)

total_characters = (
    metadata_df["character_count"].sum()
    if not metadata_df.empty
    else 0
)

successful = (
    metadata_df["status"]
    .isin(["success", "cached"])
    .sum()
)

duplicates = (
    metadata_df["status"] == "duplicate"
).sum()

failed = len(error_df)

print("=" * 60)
print("FINAL KNOWLEDGE BASE VALIDATION")
print("=" * 60)

print(f"Search results          : {len(articles_df)}")
print(f"TXT files               : {len(txt_files)}")
print(f"Metadata records        : {len(metadata_df)}")
print(f"Successful / cached     : {successful}")
print(f"Duplicate articles      : {duplicates}")
print(f"Failed downloads        : {failed}")
print(f"Total words             : {total_words:,}")
print(f"Total characters        : {total_characters:,}")
print(
    f"Files without metadata  : "
    f"{len(files_without_metadata)}"
)
print(
    f"Metadata without files  : "
    f"{len(metadata_without_files)}"
)

print("=" * 60)

if (
    len(files_without_metadata) == 0
    and len(metadata_without_files) == 0
):
    print("VALIDATION PASSED")
else:
    print("VALIDATION FAILED")

print("=" * 60)

FINAL KNOWLEDGE BASE VALIDATION
Search results          : 127
TXT files               : 126
Metadata records        : 127
Successful / cached     : 127
Duplicate articles      : 0
Failed downloads        : 0
Total words             : 356,459
Total characters        : 2,564,728
Files without metadata  : 0
Metadata without files  : 1
VALIDATION FAILED
